<a href="https://colab.research.google.com/github/k632412560046/Datathon-Dibi-/blob/main/Datathon_%7C_Dibi_%7C_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c datathon-2026-round-1

import zipfile

with zipfile.ZipFile("datathon-2026-round-1.zip", 'r') as zip_ref:
  zip_ref.extractall("data")


Saving kaggle.json to kaggle.json
100% 27.7M/27.7M [00:02<00:00, 13.2MB/s]



In [ ]:
def clean_transaction_data (orders, order_items, payments, shipments, returns, reviews):
  import pandas as pd
  import numpy as np
  def log_change(df_before, df_after, name, rule):
    removed = len(df_before) - len(df_after)

  # 1. ORDERS
  df = orders.copy()
  df.columns = df.columns.str.strip().str.lower()
  before = df.copy()
  df = df.drop_duplicates()
  log_change(before, df, "ORDERS", "drop duplicates")
  if "order_date" in df.columns:
    before_null = df["order_date"].isna().sum()
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    after_null = df["order_date"].isna().sum()

  for col in ["order_status", "payment_method", "device_type", "order_source"]:
    if col in df.columns:
      df[col] = df[col].astype(str).str.strip().str.lower()

  orders = df

  # 2. ORDER_ITEMS
  df = order_items.copy()
  df.columns = df.columns.str.strip().str.lower()
  for col in ["quantity", "unit_price", "discount_amount"]:
    if col in df.columns:
      df[col] = pd.to_numeric(df[col], errors="coerce")

  before = df.copy()
  df = df.dropna(subset=["order_id", "product_id"])
  log_change(before, df, "ORDER_ITEMS", "drop null keys")

  before = df.copy()
  df = df.drop_duplicates(subset=["order_id", "product_id"])
  log_change(before, df, "ORDER_ITEMS", "drop duplicates")

  before = df.copy()
  df = df[df["quantity"] > 0]
  log_change(before, df, "ORDER_ITEMS", "remove quantity <= 0")

  order_items = df

  # 3. PAYMENTS
  df = payments.copy()
  df.columns = df.columns.str.strip().str.lower()
  df["payment_value"] = pd.to_numeric(df["payment_value"], errors="coerce")

  before = df.copy()
  df = df.drop_duplicates(subset=["order_id"])
  log_change(before, df, "PAYMENTS", "1 order = 1 payment")

  before = df.copy()
  df = df[df["installments"] >= 1]
  log_change(before, df, "PAYMENTS", "remove invalid installments")

  payments = df

  # 4. SHIPMENTS
  df = shipments.copy()

  df.columns = df.columns.str.strip().str.lower()
  df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")
  df["delivery_date"] = pd.to_datetime(df["delivery_date"], errors="coerce")

  before = df.copy()
  df = df.drop_duplicates(subset=["order_id"])
  log_change(before, df, "SHIPMENTS", "1 shipment per order")

  before = df.copy()
  df = df[
    (df["delivery_date"].isna()) |
    (df["delivery_date"] >= df["ship_date"])
  ]
  log_change(before, df, "SHIPMENTS", "invalid date logic")

  shipments = df

  # 5. RETURNS
  df = returns.copy()

  df.columns = df.columns.str.strip().str.lower()
  df["return_date"] = pd.to_datetime(df["return_date"], errors="coerce")

  before = df.copy()
  df = df.drop_duplicates(subset=["return_id"])
  log_change(before, df, "RETURNS", "drop duplicates")

  before = df.copy()
  df = df[df["return_quantity"] > 0]
  log_change(before, df, "RETURNS", "remove quantity <= 0")

  returns = df

  # 6. REVIEWS
  df = reviews.copy()

  df.columns = df.columns.str.strip().str.lower()
  df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")

  before = df.copy()
  df = df.drop_duplicates(subset=["review_id"])
  log_change(before, df, "REVIEWS", "drop duplicates")

  before = df.copy()
  df = df[df["rating"].between(1, 5)]
  log_change(before, df, "REVIEWS", "invalid rating")

  reviews = df

  return orders, order_items, payments, shipments, returns, reviews

In [ ]:
import pandas as pd

# LOAD DATA
orders = pd.read_csv("/content/data/orders.csv")
order_items = pd.read_csv("/content/data/order_items.csv")
payments = pd.read_csv("/content/data/payments.csv")
shipments = pd.read_csv("/content/data/shipments.csv")
returns = pd.read_csv("/content/data/returns.csv")
reviews = pd.read_csv("/content/data/reviews.csv")
customers = pd.read_csv("/content/data/customers.csv")
products = pd.read_csv("/content/data/products.csv")
shipments = pd.read_csv("/content/data/shipments.csv")
geography = pd.read_csv("/content/data/geography.csv")
promotions = pd.read_csv("/content/data/promotions.csv")
sales = pd.read_csv("/content/data/sales.csv")
inventory = pd.read_csv("/content/data/inventory.csv")
web_traffic = pd.read_csv("/content/data/web_traffic.csv")
# CLEAN
orders, order_items, payments, shipments, returns, reviews = clean_transaction_data(
    orders, order_items, payments, shipments, returns, reviews
)

/tmp/ipykernel_5850/301100458.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("/content/data/order_items.csv")


In [ ]:
import pandas as pd
import numpy as np

def clean_inventory(df):
  df = df.copy()
  df.columns = df.columns.str.strip().str.lower()

  # 2. Parse date
  df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], errors='coerce')

  # 3. Remove null keys
  df = df.dropna(subset=['snapshot_date', 'product_id'])

  # 4. Remove duplicates
  df = df.drop_duplicates(subset=['snapshot_date', 'product_id'])

  # 5. Numeric cleaning
  num_cols = [
    'stock_on_hand', 'units_received', 'units_sold',
    'stockout_days', 'days_of_supply',
    'fill_rate', 'sell_through_rate'
  ]
  for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

  # 6. Remove negative values
  df = df[
    (df['stock_on_hand'] >= 0) &
    (df['units_received'] >= 0) &
    (df['units_sold'] >= 0)
  ]

  # 7. Clamp ratio columns
  df['fill_rate'] = df['fill_rate'].clip(0, 1)
  df['sell_through_rate'] = df['sell_through_rate'].clip(0, 1)

  # 8. Fix flags
  flag_cols = ['stockout_flag', 'overstock_flag', 'reorder_flag']
  for col in flag_cols:
    df[col] = df[col].apply(lambda x: 1 if x == 1 else 0)

  # 9. Fix year/month consistency
  df['year'] = df['snapshot_date'].dt.year
  df['month'] = df['snapshot_date'].dt.month

  # 10. Validate stockout_days
  days_in_month = df['snapshot_date'].dt.days_in_month
  df['stockout_days'] = np.minimum(df['stockout_days'], days_in_month)

  # 11. Drop rows missing product metadata
  df = df.dropna(subset=['product_name', 'category', 'segment'])

  return df

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression

import lightgbm as lgb
import xgboost as xgb

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# 0. SEED

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BASE = "/content/data"


# 1. SAFE LOADERS

def safe_read_csv(path, **kwargs):
    if os.path.exists(path):
        return pd.read_csv(path, **kwargs)
    return None

def ensure_datetime(df, cols):
    if df is None:
        return None
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

def safe_lower(x):
    return str(x).strip().lower() if pd.notna(x) else ""

def safe_mean(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().any():
        return float(series.mean())
    return 0.0

def safe_sum(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().any():
        return float(series.sum())
    return 0.0

def safe_std(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().sum() > 1:
        return float(series.std(ddof=0))
    return 0.0


# 2. LOAD DATA

print("=" * 70)
print("STEP 1: LOAD DATA")
print("=" * 70)

df_all = pd.read_csv(f"{BASE}/sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df_submit = pd.read_csv(f"{BASE}/sample_submission.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

df_web = safe_read_csv(f"{BASE}/web_traffic.csv", parse_dates=["date"])
df_inv = safe_read_csv(f"{BASE}/inventory.csv", parse_dates=["snapshot_date"])
df_promo = safe_read_csv(f"{BASE}/promotions.csv", parse_dates=["start_date", "end_date"])

# Transaction/master tables (may or may not exist depending on your folder)
df_products   = safe_read_csv(f"{BASE}/products.csv")
df_customers  = safe_read_csv(f"{BASE}/customers.csv")
df_geo        = safe_read_csv(f"{BASE}/geography.csv")
df_orders     = safe_read_csv(f"{BASE}/orders.csv", parse_dates=["order_date"])
df_items      = safe_read_csv(f"{BASE}/order_items.csv")
df_payments   = safe_read_csv(f"{BASE}/payments.csv")
df_shipments  = safe_read_csv(f"{BASE}/shipments.csv", parse_dates=["ship_date", "delivery_date"])
df_returns    = safe_read_csv(f"{BASE}/returns.csv", parse_dates=["return_date"])
df_reviews    = safe_read_csv(f"{BASE}/reviews.csv", parse_dates=["review_date"])

df_train = df_all[df_all["Date"] <= "2021-12-31"].copy().reset_index(drop=True)
df_val   = df_all[(df_all["Date"] >= "2022-01-01") & (df_all["Date"] <= "2022-12-31")].copy().reset_index(drop=True)
df_full  = df_all[df_all["Date"] <= "2022-12-31"].copy().reset_index(drop=True)

print(f"  Train (<=2021): {len(df_train):,}")
print(f"  Val   (2022)  : {len(df_val):,}")
print(f"  Full  (<=2022): {len(df_full):,}")
print(f"  Test skeleton : {len(df_submit):,}")


# 3. LOOKUPS: WEB / INVENTORY / PROMO

print("\n" + "=" * 70)
print("STEP 2: EXTERNAL LOOKUPS")
print("=" * 70)

def build_web_lookup(df_web):
    if df_web is None or len(df_web) == 0:
        return {}, {}, {}
    need_cols = {"date", "sessions", "bounce_rate"}
    if not need_cols.issubset(df_web.columns):
        return {}, {}, {}

    daily = df_web.groupby("date").agg(
        sessions=("sessions", "sum"),
        unique_visitors=("unique_visitors", "sum") if "unique_visitors" in df_web.columns else ("sessions", "sum"),
        page_views=("page_views", "sum") if "page_views" in df_web.columns else ("sessions", "sum"),
        bounce_rate=("bounce_rate", "mean"),
        avg_session_duration_sec=("avg_session_duration_sec", "mean") if "avg_session_duration_sec" in df_web.columns else ("sessions", "mean"),
    ).reset_index().rename(columns={"date": "Date"}).sort_values("Date")

    # lags / rolling
    daily["sessions_lag7"] = daily["sessions"].shift(7)
    daily["sessions_lag14"] = daily["sessions"].shift(14)
    daily["sessions_roll7"] = daily["sessions"].shift(1).rolling(7, min_periods=1).mean()
    daily["sessions_roll30"] = daily["sessions"].shift(1).rolling(30, min_periods=1).mean()

    daily["bounce_lag7"] = daily["bounce_rate"].shift(7)
    daily["bounce_roll7"] = daily["bounce_rate"].shift(1).rolling(7, min_periods=1).mean()
    daily["bounce_roll30"] = daily["bounce_rate"].shift(1).rolling(30, min_periods=1).mean()

    daily["traffic_eff"] = daily["sessions_roll7"] / (daily["sessions_roll30"] + 1.0)
    daily["traffic_quality"] = (daily["page_views"] / (daily["unique_visitors"] + 1.0)) * (
        daily["avg_session_duration_sec"] / (daily["bounce_rate"] + 0.05)
    )

    daily["month"] = daily["Date"].dt.month
    daily["dow"] = daily["Date"].dt.dayofweek

    num_cols = [c for c in daily.columns if c not in ["Date"]]
    daily = daily.set_index("Date")

    daily_lookup = daily[num_cols].to_dict("index")
    month_lookup = daily.groupby("month")[num_cols].mean(numeric_only=True).to_dict("index")
    dow_lookup = daily.groupby("dow")[num_cols].mean(numeric_only=True).to_dict("index")
    md_lookup = daily.groupby(["month", "dow"])[num_cols].mean(numeric_only=True).to_dict("index")
    return daily_lookup, month_lookup, dow_lookup

def build_inventory_lookup(df_inv):
    if df_inv is None or len(df_inv) == 0:
        return {}, {}, {}

    df = df_inv.copy()

    if "snapshot_date" not in df.columns:
        return {}, {}, {}

    df["year"] = df["snapshot_date"].dt.year
    df["month"] = df["snapshot_date"].dt.month

    agg_dict = {
        "avg_stock": ("stock_on_hand", "mean") if "stock_on_hand" in df.columns else ("year", "mean"),
        "fill_rate": ("fill_rate", "mean") if "fill_rate" in df.columns else ("year", "mean"),
        "pct_overstock": ("overstock_flag", "mean") if "overstock_flag" in df.columns else ("year", "mean"),
        "sell_through": ("sell_through_rate", "mean") if "sell_through_rate" in df.columns else ("year", "mean"),
        "stockout_days": ("stockout_days", "mean") if "stockout_days" in df.columns else ("year", "mean"),
        "days_of_supply": ("days_of_supply", "mean") if "days_of_supply" in df.columns else ("year", "mean"),
    }

    inv_m = df.groupby(["year", "month"]).agg(**agg_dict).reset_index().sort_values(["year", "month"])
    inv_m["ym"] = inv_m["year"] * 100 + inv_m["month"]

    for c in ["avg_stock", "fill_rate", "pct_overstock", "sell_through", "stockout_days", "days_of_supply"]:
        inv_m[f"{c}_lag1m"] = inv_m[c].shift(1)
        inv_m[f"{c}_lag3m"] = inv_m[c].shift(3)

    inv_m["inventory_health"] = (
        inv_m["fill_rate_lag1m"].fillna(inv_m["fill_rate"]) *
        inv_m["sell_through_lag1m"].fillna(inv_m["sell_through"]) *
        (1.0 - inv_m["pct_overstock_lag1m"].fillna(inv_m["pct_overstock"]))
    )
    inv_m["stock_pressure"] = inv_m["avg_stock_lag1m"].fillna(inv_m["avg_stock"]) / (inv_m["sell_through_lag1m"].fillna(inv_m["sell_through"]) + 0.05)
    inv_m["overstock_risk"] = inv_m["pct_overstock_lag1m"].fillna(inv_m["pct_overstock"]) * inv_m["avg_stock_lag1m"].fillna(inv_m["avg_stock"])

    # IMPORTANT: exclude year, month, ym from columns before setting ym as index
    num_cols = [c for c in inv_m.columns if c not in ["year", "month", "ym"]]

    inv_lookup = inv_m.set_index("ym")[num_cols].to_dict("index")
    month_lookup = inv_m.groupby("month")[num_cols].mean(numeric_only=True).to_dict("index")
    dow_lookup = {}

    return inv_lookup, month_lookup, dow_lookup

def build_promo_lookup(df_promo, min_date, max_date):
    if df_promo is None or len(df_promo) == 0:
        return {}, {}, {}
    if not {"start_date", "end_date"}.issubset(df_promo.columns):
        return {}, {}, {}

    rows = []
    for d in pd.date_range(min_date, max_date, freq="D"):
        active = df_promo[(df_promo["start_date"] <= d) & (df_promo["end_date"] >= d)]
        rows.append({
            "Date": d,
            "n_promos": len(active),
            "avg_discount": safe_mean(active["discount_value"]) if "discount_value" in active.columns else 0.0,
            "max_discount": float(active["discount_value"].max()) if ("discount_value" in active.columns and len(active) > 0) else 0.0,
            "has_promo": int(len(active) > 0),
            "stackable_rate": safe_mean(active["stackable_flag"]) if "stackable_flag" in active.columns else 0.0,
            "min_order_value": safe_mean(active["min_order_value"]) if "min_order_value" in active.columns else 0.0,
        })
    daily = pd.DataFrame(rows)
    daily["promo_intensity"] = daily["n_promos"] * daily["avg_discount"]
    daily["promo_stack_intensity"] = daily["n_promos"] * daily["stackable_rate"]
    daily["month"] = daily["Date"].dt.month
    daily["dow"] = daily["Date"].dt.dayofweek

    num_cols = [c for c in daily.columns if c not in ["Date"]]
    daily_lookup = daily.set_index("Date")[num_cols].to_dict("index")
    month_lookup = daily.groupby("month")[num_cols].mean(numeric_only=True).to_dict("index")
    dow_lookup = daily.groupby("dow")[num_cols].mean(numeric_only=True).to_dict("index")
    md_lookup = daily.groupby(["month", "dow"])[num_cols].mean(numeric_only=True).to_dict("index")
    return daily_lookup, month_lookup, dow_lookup

web_daily_lookup, web_month_lookup, web_dow_lookup = build_web_lookup(df_web)
inv_daily_lookup, inv_month_lookup, inv_dow_lookup = build_inventory_lookup(df_inv)
promo_daily_lookup, promo_month_lookup, promo_dow_lookup = build_promo_lookup(
    df_promo, df_all["Date"].min(), df_submit["Date"].max()
)

print("   External signals ready")


# 4. BUSINESS / TRANSACTION PROFILE LOOKUPS

print("\n" + "=" * 70)
print("STEP 3: BUSINESS PROFILES")
print("=" * 70)

def build_business_profile_lookups(
    products, customers, geography, orders, order_items, payments, shipments, returns, reviews
):
    if orders is None or order_items is None or products is None:
        return {}, {}, {}

    # Minimal cleaning
    orders = orders.copy()
    order_items = order_items.copy()
    products = products.copy()

    if "order_date" not in orders.columns:
        return {}, {}, {}
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

    # Safe product subset
    prod_cols = [c for c in ["product_id", "category", "segment", "size", "price", "cogs"] if c in products.columns]
    prod = products[prod_cols].copy()

    items = order_items.merge(prod, on="product_id", how="left")

    # Item-level engineered fields
    if "quantity" not in items.columns:
        items["quantity"] = 1
    if "unit_price" not in items.columns:
        items["unit_price"] = items["price"] if "price" in items.columns else 0.0
    if "discount_amount" not in items.columns:
        items["discount_amount"] = 0.0

    items["gross_item_value"] = items["quantity"] * items["unit_price"]
    items["net_item_value"] = items["gross_item_value"] - items["discount_amount"]
    items["promo_item"] = 0
    if "promo_id" in items.columns:
        items["promo_item"] = items["promo_id"].notna().astype(int)
    if "promo_id_2" in items.columns:
        items["promo_item"] = ((items["promo_item"] == 1) | (items["promo_id_2"].notna())).astype(int)

    # Key EDA category/segment flags
    def flag_col(df, col, value):
        if col not in df.columns:
            return np.zeros(len(df), dtype=int)
        return (df[col].map(safe_lower) == safe_lower(value)).astype(int).values

    items["streetwear_flag"] = flag_col(items, "category", "streetwear")
    items["casual_flag"]     = flag_col(items, "segment", "casual")
    items["premium_flag"]    = flag_col(items, "segment", "premium")
    items["performance_flag"]= flag_col(items, "segment", "performance")
    items["activewear_flag"] = flag_col(items, "segment", "activewear")
    items["standard_flag"]   = flag_col(items, "segment", "standard")

    if "size" in items.columns:
        items["size_s"]  = (items["size"].map(safe_lower) == "s").astype(int)
        items["size_m"]  = (items["size"].map(safe_lower) == "m").astype(int)
        items["size_l"]  = (items["size"].map(safe_lower) == "l").astype(int)
        items["size_xl"] = (items["size"].map(safe_lower) == "xl").astype(int)
    else:
        items["size_s"] = items["size_m"] = items["size_l"] = items["size_xl"] = 0

    # Aggregate order_items to order level
    order_item_agg = items.groupby("order_id").agg(
        order_items_cnt=("product_id", "count"),
        qty_sum=("quantity", "sum"),
        gross_value=("gross_item_value", "sum"),
        discount_sum=("discount_amount", "sum"),
        net_value=("net_item_value", "sum"),
        unit_price_mean=("unit_price", "mean"),
        unit_price_std=("unit_price", "std"),
        promo_item_share=("promo_item", "mean"),
        streetwear_share=("streetwear_flag", "mean"),
        casual_share=("casual_flag", "mean"),
        premium_share=("premium_flag", "mean"),
        performance_share=("performance_flag", "mean"),
        activewear_share=("activewear_flag", "mean"),
        standard_share=("standard_flag", "mean"),
        size_s_share=("size_s", "mean"),
        size_m_share=("size_m", "mean"),
        size_l_share=("size_l", "mean"),
        size_xl_share=("size_xl", "mean"),
    ).reset_index()

    # Orders base
    ord_base_cols = [c for c in ["order_id", "order_date", "customer_id", "zip", "order_status", "payment_method", "device_type", "order_source"] if c in orders.columns]
    ord = orders[ord_base_cols].copy()

    # Customer / geography
    if customers is not None and "customer_id" in customers.columns:
        cust_cols = [c for c in ["customer_id", "zip", "city", "signup_date", "gender", "age_group", "acquisition_channel"] if c in customers.columns]
        ord = ord.merge(customers[cust_cols].copy(), on="customer_id", how="left", suffixes=("", "_cust"))
    if geography is not None and "zip" in geography.columns:
        geo_cols = [c for c in ["zip", "city", "region", "district"] if c in geography.columns]
        ord = ord.merge(geography[geo_cols].copy(), on="zip", how="left", suffixes=("", "_geo"))

    # Payments
    if payments is not None and "order_id" in payments.columns:
        pay_cols = [c for c in ["order_id", "payment_method", "payment_value", "installments"] if c in payments.columns]
        pay = payments[pay_cols].copy()
        pay_agg = pay.groupby("order_id").agg(
            payment_value=("payment_value", "mean") if "payment_value" in pay.columns else ("order_id", "count"),
            installments=("installments", "mean") if "installments" in pay.columns else ("order_id", "count"),
        ).reset_index()
    else:
        pay_agg = pd.DataFrame({"order_id": ord["order_id"].values, "payment_value": 0.0, "installments": 0.0})

    # Shipments
    if shipments is not None and "order_id" in shipments.columns:
        shp = shipments.copy()
        if {"ship_date", "delivery_date"}.issubset(shp.columns):
            shp["ship_delay_days"] = (shp["delivery_date"] - shp["ship_date"]).dt.days
        else:
            shp["ship_delay_days"] = np.nan
        shp_cols = [c for c in ["order_id", "ship_delay_days", "shipping_fee"] if c in shp.columns]
        shp_agg = shp[shp_cols].groupby("order_id").agg(
            ship_delay_days=("ship_delay_days", "mean"),
            shipping_fee=("shipping_fee", "mean") if "shipping_fee" in shp.columns else ("order_id", "count"),
        ).reset_index()
    else:
        shp_agg = pd.DataFrame({"order_id": ord["order_id"].values, "ship_delay_days": 0.0, "shipping_fee": 0.0})

    # Returns
    if returns is not None and "order_id" in returns.columns:
        ret = returns.copy()
        ret["return_flag"] = 1
        if "return_reason" in ret.columns:
            ret["wrong_size_flag"] = (ret["return_reason"].map(safe_lower) == "wrong_size").astype(int)
            ret["changed_mind_flag"] = (ret["return_reason"].map(safe_lower) == "changed_mind").astype(int)
            ret["defective_flag"] = (ret["return_reason"].map(safe_lower) == "defective").astype(int)
        else:
            ret["wrong_size_flag"] = ret["changed_mind_flag"] = ret["defective_flag"] = 0

        ret_cols = [c for c in ["order_id", "return_quantity", "refund_amount", "return_flag", "wrong_size_flag", "changed_mind_flag", "defective_flag"] if c in ret.columns]
        ret_agg = ret[ret_cols].groupby("order_id").agg(
            return_quantity=("return_quantity", "sum") if "return_quantity" in ret.columns else ("order_id", "count"),
            refund_amount=("refund_amount", "sum") if "refund_amount" in ret.columns else ("order_id", "count"),
            return_flag=("return_flag", "max"),
            wrong_size_rate=("wrong_size_flag", "mean"),
            changed_mind_rate=("changed_mind_flag", "mean"),
            defective_rate=("defective_flag", "mean"),
        ).reset_index()
    else:
        ret_agg = pd.DataFrame({"order_id": ord["order_id"].values, "return_quantity": 0.0, "refund_amount": 0.0, "return_flag": 0, "wrong_size_rate": 0.0, "changed_mind_rate": 0.0, "defective_rate": 0.0})

    # Reviews
    if reviews is not None and "order_id" in reviews.columns:
        rv = reviews.copy()
        rv_cols = [c for c in ["order_id", "rating"] if c in rv.columns]
        rv_agg = rv[rv_cols].groupby("order_id").agg(
            rating_mean=("rating", "mean") if "rating" in rv.columns else ("order_id", "count"),
            rating_std=("rating", "std") if "rating" in rv.columns else ("order_id", "count"),
        ).reset_index()
    else:
        rv_agg = pd.DataFrame({"order_id": ord["order_id"].values, "rating_mean": 0.0, "rating_std": 0.0})

    # Merge to order-level table
    od = ord.merge(order_item_agg, on="order_id", how="left")
    od = od.merge(pay_agg, on="order_id", how="left")
    od = od.merge(shp_agg, on="order_id", how="left")
    od = od.merge(ret_agg, on="order_id", how="left")
    od = od.merge(rv_agg, on="order_id", how="left")

    # Customer / geo derived flags
    if "age_group" in od.columns:
        od["genz_flag"] = (od["age_group"].map(safe_lower).isin(["gen z", "genz", "18-24", "18–24"])).astype(int)
    else:
        od["genz_flag"] = 0
    if "region" in od.columns:
        od["west_flag"] = (od["region"].map(safe_lower) == "west").astype(int)
        od["central_flag"] = (od["region"].map(safe_lower) == "central").astype(int)
        od["east_flag"] = (od["region"].map(safe_lower) == "east").astype(int)
    else:
        od["west_flag"] = od["central_flag"] = od["east_flag"] = 0

    if "order_status" in od.columns:
        od["cancel_flag"] = (od["order_status"].map(safe_lower) == "cancelled").astype(int)
    else:
        od["cancel_flag"] = 0

    if "order_source" in od.columns:
        od["social_flag"] = (od["order_source"].map(safe_lower).str.contains("social")).astype(int)
        od["organic_flag"] = (od["order_source"].map(safe_lower).str.contains("organic")).astype(int)
    else:
        od["social_flag"] = od["organic_flag"] = 0

    if "payment_method" in od.columns:
        od["cod_flag"] = (od["payment_method"].map(safe_lower) == "cod").astype(int)
        od["card_flag"] = (od["payment_method"].map(safe_lower).isin(["credit_card", "debit_card"])).astype(int)
    else:
        od["cod_flag"] = od["card_flag"] = 0

    # Daily aggregate from order-level table
    od["order_date"] = pd.to_datetime(od["order_date"], errors="coerce")
    od = od.dropna(subset=["order_date"]).copy()

    # Fill missing numeric with zeros so aggregation is stable
    numeric_cols = od.select_dtypes(include=[np.number]).columns.tolist()
    od[numeric_cols] = od[numeric_cols].fillna(0.0)

    daily = od.groupby("order_date").agg(
        n_orders=("order_id", "count"),
        n_customers=("customer_id", pd.Series.nunique) if "customer_id" in od.columns else ("order_id", "count"),
        order_items_cnt=("order_items_cnt", "sum"),
        qty_sum=("qty_sum", "sum"),
        gross_value=("gross_value", "sum"),
        discount_sum=("discount_sum", "sum"),
        net_value=("net_value", "sum"),
        unit_price_mean=("unit_price_mean", "mean"),
        unit_price_std=("unit_price_std", "mean"),
        promo_item_share=("promo_item_share", "mean"),
        streetwear_share=("streetwear_share", "mean"),
        casual_share=("casual_share", "mean"),
        premium_share=("premium_share", "mean"),
        performance_share=("performance_share", "mean"),
        activewear_share=("activewear_share", "mean"),
        standard_share=("standard_share", "mean"),
        size_s_share=("size_s_share", "mean"),
        size_m_share=("size_m_share", "mean"),
        size_l_share=("size_l_share", "mean"),
        size_xl_share=("size_xl_share", "mean"),
        payment_value=("payment_value", "mean"),
        installments=("installments", "mean"),
        ship_delay_days=("ship_delay_days", "mean"),
        shipping_fee=("shipping_fee", "mean"),
        return_quantity=("return_quantity", "sum"),
        refund_amount=("refund_amount", "sum"),
        return_flag=("return_flag", "mean"),
        wrong_size_rate=("wrong_size_rate", "mean"),
        changed_mind_rate=("changed_mind_rate", "mean"),
        defective_rate=("defective_rate", "mean"),
        rating_mean=("rating_mean", "mean"),
        rating_std=("rating_std", "mean"),
        genz_flag=("genz_flag", "mean"),
        west_flag=("west_flag", "mean"),
        central_flag=("central_flag", "mean"),
        east_flag=("east_flag", "mean"),
        cancel_flag=("cancel_flag", "mean"),
        social_flag=("social_flag", "mean"),
        organic_flag=("organic_flag", "mean"),
        cod_flag=("cod_flag", "mean"),
        card_flag=("card_flag", "mean"),
    ).reset_index()

    daily["month"] = daily["order_date"].dt.month
    daily["dow"] = daily["order_date"].dt.dayofweek
    daily["month_dow"] = daily["month"].astype(str) + "_" + daily["dow"].astype(str)

    # Derived business ratios
    daily["discount_rate"] = daily["discount_sum"] / (daily["gross_value"] + 1e-9)
    daily["net_margin_proxy"] = (daily["gross_value"] - daily["discount_sum"]) / (daily["gross_value"] + 1e-9)
    daily["streetwear_dominance"] = daily["streetwear_share"] - daily["casual_share"]
    daily["product_mix_entropy"] = -(
        np.nan_to_num(daily["streetwear_share"]) * np.log(np.nan_to_num(daily["streetwear_share"]) + 1e-9)
        + np.nan_to_num(daily["casual_share"]) * np.log(np.nan_to_num(daily["casual_share"]) + 1e-9)
        + np.nan_to_num(daily["premium_share"]) * np.log(np.nan_to_num(daily["premium_share"]) + 1e-9)
    )
    daily["return_pressure"] = daily["return_flag"] * (1.0 + daily["wrong_size_rate"] + daily["changed_mind_rate"])
    daily["conversion_proxy"] = daily["n_orders"] / (daily["qty_sum"] + 1e-9)
    daily["inventory_pressure_proxy"] = daily["return_flag"] + daily["cancel_flag"]
    daily["promo_penetration_proxy"] = daily["promo_item_share"] * daily["discount_rate"]

    num_cols = [c for c in daily.columns if c not in ["order_date", "month_dow"]]
    month_lookup = daily.groupby("month")[num_cols].mean(numeric_only=True).to_dict("index")
    dow_lookup = daily.groupby("dow")[num_cols].mean(numeric_only=True).to_dict("index")
    md_lookup = daily.groupby(["month", "dow"])[num_cols].mean(numeric_only=True).to_dict("index")

    return month_lookup, dow_lookup, md_lookup

tx_month_lookup, tx_dow_lookup, tx_md_lookup = build_business_profile_lookups(
    df_products, df_customers, df_geo, df_orders, df_items, df_payments, df_shipments, df_returns, df_reviews
)

print("   Business profiles ready")


# 5. FEATURE HELPERS

print("\n" + "=" * 70)
print("STEP 4: FEATURE HELPERS")
print("=" * 70)

LAGS = [1, 2, 3, 5, 7, 14, 21, 28, 60, 90, 180, 365, 366]
ROLLS = [7, 14, 30, 90, 365]

def weighted_profile_value(date, feature_name, md_lookup, m_lookup, d_lookup, default=0.0):
    date = pd.Timestamp(date)
    m = int(date.month)
    d = int(date.dayofweek)

    vals = []
    weights = []

    v_md = md_lookup.get((m, d), {}).get(feature_name, np.nan)
    v_m  = m_lookup.get(m, {}).get(feature_name, np.nan)
    v_d  = d_lookup.get(d, {}).get(feature_name, np.nan)

    if pd.notna(v_md):
        vals.append(v_md); weights.append(0.50)
    if pd.notna(v_m):
        vals.append(v_m); weights.append(0.30)
    if pd.notna(v_d):
        vals.append(v_d); weights.append(0.20)

    if len(vals) == 0:
        return default
    return float(np.average(vals, weights=weights))

def seasonal_naive_base(history):
    """
    Strong seasonal baseline using same-day last year + short/medium seasonality.
    """
    arr = np.asarray(history, dtype=float)
    if len(arr) == 0:
        return 0.0

    comps = []
    weights = []

    def add_if_exists(k, w):
        if len(arr) >= k and np.isfinite(arr[-k]):
            comps.append(arr[-k])
            weights.append(w)

    add_if_exists(365, 0.35)
    add_if_exists(366, 0.10)
    add_if_exists(364, 0.10)
    add_if_exists(7,   0.15)
    add_if_exists(14,  0.05)
    add_if_exists(30,  0.15)
    add_if_exists(90,  0.10)

    if len(comps) > 0:
        base = float(np.average(comps, weights=weights))
    else:
        base = float(np.mean(arr))

    if len(arr) >= 30:
        cap = max(np.max(arr[-365:]), 1.0) * 1.08
        base = min(base, cap)

    return max(base, 0.0)

def safe_slope(y):
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 5:
        return np.nan
    x = np.arange(n, dtype=float)
    x -= x.mean()
    denom = np.sum(x * x)
    if denom == 0:
        return np.nan
    return float(np.sum((y - y.mean()) * x) / denom)

def date_features(date):
    date = pd.Timestamp(date)
    y, m, d = date.year, date.month, date.day
    dow = date.dayofweek
    doy = date.dayofyear

    feats = {
        "year": y,
        "month": m,
        "day": d,
        "dayofweek": dow,
        "dayofyear": doy,
        "quarter": date.quarter,
        "weekofyear": int(date.isocalendar().week),
        "is_weekend": int(dow >= 5),
        "is_month_start": int(date.is_month_start),
        "is_month_end": int(date.is_month_end),
        "is_quarter_start": int(date.is_quarter_start),
        "is_quarter_end": int(date.is_quarter_end),
        "is_year_start": int(date.is_year_start),
        "is_year_end": int(date.is_year_end),
        "days_to_month_end": int((date + pd.offsets.MonthEnd(0) - date).days),
        "days_since_month_start": int((date - date.replace(day=1)).days),
        "is_payday": int(d in [1, 15, 20, 25, 30, 31]),
        "month_sin": np.sin(2 * np.pi * m / 12.0),
        "month_cos": np.cos(2 * np.pi * m / 12.0),
        "dow_sin": np.sin(2 * np.pi * dow / 7.0),
        "dow_cos": np.cos(2 * np.pi * dow / 7.0),
        "doy_sin": np.sin(2 * np.pi * doy / 365.0),
        "doy_cos": np.cos(2 * np.pi * doy / 365.0),
        "doy_sin2": np.sin(4 * np.pi * doy / 365.0),
        "doy_cos2": np.cos(4 * np.pi * doy / 365.0),
        "trend": (y - 2012) * 365 + doy,
    }
    feats["trend_sq"] = (feats["trend"] ** 2) / 1e6

    # regime / structural break features
    feats["is_pre2019"] = int(y < 2019)
    feats["is_covid"] = int((y >= 2019) and (y <= 2022))
    feats["is_post_covid"] = int(y >= 2023)
    feats["is_2019_2020"] = int((y >= 2019) and (y <= 2020))
    feats["is_2021_plus"] = int(y >= 2021)

    feats["trend_x_pre"] = feats["trend"] * feats["is_pre2019"]
    feats["trend_x_covid"] = feats["trend"] * feats["is_covid"]
    feats["trend_x_post"] = feats["trend"] * feats["is_post_covid"]

    feats["is_tet"] = int(((m == 1) and (d >= 15)) or ((m == 2) and (d <= 20)))
    feats["is_peak_month"] = int(m == 5)
    feats["is_pre_peak"] = int(m in [3, 4])
    feats["is_yearend"] = int(m in [11, 12])
    feats["months_to_peak"] = min(abs(m - 5), 12 - abs(m - 5))
    feats["is_monthly_camp"] = int(
        ((m == 8) and (d == 8)) or ((m == 9) and (d == 9)) or
        ((m == 10) and (d == 10)) or ((m == 11) and (d == 11)) or
        ((m == 12) and (d == 12))
    )
    return feats

def history_features(history):
    arr = np.asarray(history, dtype=float)
    out = {}

    def safe_lag(k):
        return float(arr[-k]) if len(arr) >= k else np.nan

    for lag in LAGS:
        out[f"lag_{lag}"] = safe_lag(lag)

    for w in ROLLS:
        if len(arr) == 0:
            out[f"roll_mean_{w}"] = np.nan
            out[f"roll_std_{w}"] = np.nan
            out[f"roll_median_{w}"] = np.nan
            out[f"roll_min_{w}"] = np.nan
            out[f"roll_max_{w}"] = np.nan
            out[f"roll_range_{w}"] = np.nan
            out[f"ewm_{w}"] = np.nan
            out[f"slope_{w}"] = np.nan
        else:
            tail = arr[-w:] if len(arr) >= w else arr
            out[f"roll_mean_{w}"] = float(np.mean(tail))
            out[f"roll_std_{w}"] = float(np.std(tail, ddof=0)) if len(tail) > 1 else 0.0
            out[f"roll_median_{w}"] = float(np.median(tail))
            out[f"roll_min_{w}"] = float(np.min(tail))
            out[f"roll_max_{w}"] = float(np.max(tail))
            out[f"roll_range_{w}"] = float(np.max(tail) - np.min(tail))
            alpha = 2.0 / (w + 1.0)
            ewm = tail[0]
            for v in tail[1:]:
                ewm = alpha * v + (1 - alpha) * ewm
            out[f"ewm_{w}"] = float(ewm)
            out[f"slope_{w}"] = safe_slope(tail)

    out["yoy_diff"] = float(arr[-1] - arr[-366]) if len(arr) >= 366 else np.nan
    out["yoy_ratio"] = float(arr[-1] / (arr[-366] + 1e-9)) if len(arr) >= 366 else np.nan

    out["growth_7"] = float(arr[-1] - arr[-8]) if len(arr) >= 8 else np.nan
    out["growth_30"] = float(arr[-1] - arr[-31]) if len(arr) >= 31 else np.nan

    if len(arr) >= 30:
        tail30 = arr[-30:]
        out["roll_max_30"] = float(np.max(tail30))
        out["roll_min_30"] = float(np.min(tail30))
        out["range_30"] = float(np.max(tail30) - np.min(tail30))
    else:
        out["roll_max_30"] = np.nan
        out["roll_min_30"] = np.nan
        out["range_30"] = np.nan

    base = seasonal_naive_base(arr)
    out["base_raw"] = base
    out["base_log"] = np.log1p(max(base, 0.0))

    out["base_vs_lag365"] = float(base / (safe_lag(365) + 1e-9)) if len(arr) >= 365 else np.nan
    out["base_vs_roll30"] = float(base / (out["roll_mean_30"] + 1e-9)) if pd.notna(out.get("roll_mean_30", np.nan)) else np.nan
    out["base_vs_roll365"] = float(base / (out["roll_mean_365"] + 1e-9)) if pd.notna(out.get("roll_mean_365", np.nan)) else np.nan

    out["momentum_7_30"] = float(out["roll_mean_7"] / (out["roll_mean_30"] + 1e-9)) if pd.notna(out.get("roll_mean_7", np.nan)) and pd.notna(out.get("roll_mean_30", np.nan)) else np.nan
    out["momentum_30_90"] = float(out["roll_mean_30"] / (out["roll_mean_90"] + 1e-9)) if pd.notna(out.get("roll_mean_30", np.nan)) and pd.notna(out.get("roll_mean_90", np.nan)) else np.nan
    out["volatility_30"] = float(out["roll_std_30"] / (out["roll_mean_30"] + 1e-9)) if pd.notna(out.get("roll_std_30", np.nan)) and pd.notna(out.get("roll_mean_30", np.nan)) else np.nan
    out["volatility_90"] = float(out["roll_std_90"] / (out["roll_mean_90"] + 1e-9)) if pd.notna(out.get("roll_std_90", np.nan)) and pd.notna(out.get("roll_mean_90", np.nan)) else np.nan
    return out

def external_features(date):
    date = pd.Timestamp(date)
    ym = date.year * 100 + date.month
    m = int(date.month)
    d = int(date.dayofweek)

    # actual daily if available, otherwise seasonal business profile fallback
    web_feat = web_daily_lookup.get(date, {})
    inv_feat = inv_daily_lookup.get(ym, {})
    promo_feat = promo_daily_lookup.get(date, {})

    if len(web_feat) == 0:
        web_feat = {
            k: weighted_profile_value(date, k, {}, web_month_lookup, web_dow_lookup, 0.0)
            for k in ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec",
                      "sessions_lag7", "sessions_lag14", "sessions_roll7", "sessions_roll30",
                      "bounce_lag7", "bounce_roll7", "bounce_roll30", "traffic_eff", "traffic_quality"]
        }
    if len(inv_feat) == 0:
        inv_feat = {
            k: weighted_profile_value(date, k, {}, inv_month_lookup, inv_dow_lookup, 0.0)
            for k in ["avg_stock", "fill_rate", "pct_overstock", "sell_through", "stockout_days", "days_of_supply",
                      "avg_stock_lag1m", "fill_rate_lag1m", "pct_overstock_lag1m", "sell_through_lag1m",
                      "inventory_health", "stock_pressure", "overstock_risk"]
        }
    if len(promo_feat) == 0:
        promo_feat = {
            k: weighted_profile_value(date, k, promo_md_lookup, promo_month_lookup, promo_dow_lookup, 0.0)
            for k in ["n_promos", "avg_discount", "max_discount", "has_promo", "stackable_rate", "min_order_value",
                      "promo_intensity", "promo_stack_intensity"]
        }

    tx_keys = [
        "n_orders", "n_customers", "order_items_cnt", "qty_sum", "gross_value", "discount_sum", "net_value",
        "unit_price_mean", "unit_price_std", "promo_item_share", "streetwear_share", "casual_share",
        "premium_share", "performance_share", "activewear_share", "standard_share",
        "size_s_share", "size_m_share", "size_l_share", "size_xl_share",
        "payment_value", "installments", "ship_delay_days", "shipping_fee",
        "return_quantity", "refund_amount", "return_flag", "wrong_size_rate", "changed_mind_rate", "defective_rate",
        "rating_mean", "rating_std", "genz_flag", "west_flag", "central_flag", "east_flag",
        "cancel_flag", "social_flag", "organic_flag", "cod_flag", "card_flag",
        "discount_rate", "net_margin_proxy", "streetwear_dominance", "product_mix_entropy",
        "return_pressure", "conversion_proxy", "inventory_pressure_proxy", "promo_penetration_proxy"
    ]

    tx_feat = {
        k: weighted_profile_value(date, k, tx_md_lookup, tx_month_lookup, tx_dow_lookup, 0.0)
        for k in tx_keys
    }

    # regime-aware interactions and EDA-driven derived signals
    regime = 0 if date.year < 2019 else (1 if date.year <= 2020 else 2)
    tx_feat["regime"] = regime

    feats = {
        # web
        "sessions_lag7": web_feat.get("sessions_lag7", 0.0),
        "sessions_lag14": web_feat.get("sessions_lag14", 0.0),
        "sessions_roll7": web_feat.get("sessions_roll7", 0.0),
        "sessions_roll30": web_feat.get("sessions_roll30", 0.0),
        "bounce_lag7": web_feat.get("bounce_lag7", 0.0),
        "bounce_roll7": web_feat.get("bounce_roll7", 0.0),
        "bounce_roll30": web_feat.get("bounce_roll30", 0.0),
        "traffic_eff": web_feat.get("traffic_eff", 0.0),
        "traffic_quality": web_feat.get("traffic_quality", 0.0),

        # inventory
        "avg_stock_lag1m": inv_feat.get("avg_stock_lag1m", 0.0),
        "fill_rate_lag1m": inv_feat.get("fill_rate_lag1m", 0.0),
        "pct_overstock_lag1m": inv_feat.get("pct_overstock_lag1m", 0.0),
        "sell_through_lag1m": inv_feat.get("sell_through_lag1m", 0.0),
        "stockout_days_lag1m": inv_feat.get("stockout_days_lag1m", 0.0),
        "days_of_supply_lag1m": inv_feat.get("days_of_supply_lag1m", 0.0),
        "inventory_health": inv_feat.get("inventory_health", 0.0),
        "stock_pressure": inv_feat.get("stock_pressure", 0.0),
        "overstock_risk": inv_feat.get("overstock_risk", 0.0),

        # promo
        "n_promos": promo_feat.get("n_promos", 0.0),
        "avg_discount": promo_feat.get("avg_discount", 0.0),
        "max_discount": promo_feat.get("max_discount", 0.0),
        "stackable_rate": promo_feat.get("stackable_rate", 0.0),
        "min_order_value": promo_feat.get("min_order_value", 0.0),
        "has_promo": promo_feat.get("has_promo", 0.0),
        "promo_intensity": promo_feat.get("promo_intensity", 0.0),
        "promo_stack_intensity": promo_feat.get("promo_stack_intensity", 0.0),

        # transaction/business profiles
        **tx_feat,
    }

    # strong EDA-based interactions
    feats["sessions_x_covid"] = feats["sessions_roll7"] * int((date.year >= 2019) and (date.year <= 2022))
    feats["traffic_x_regime"] = feats["sessions_roll7"] * regime
    feats["traffic_x_quality"] = feats["sessions_roll7"] * feats["traffic_quality"]
    feats["traffic_x_promo"] = feats["sessions_roll7"] * feats["promo_intensity"]
    feats["promo_x_peak"] = feats["has_promo"] * int(date.month == 5)
    feats["promo_x_yearend"] = feats["has_promo"] * int(date.month in [11, 12])
    feats["stock_x_yearend"] = feats["avg_stock_lag1m"] * int(date.month in [11, 12])
    feats["inventory_x_promo"] = feats["inventory_health"] * feats["promo_intensity"]
    feats["quality_x_promo"] = feats["traffic_quality"] * feats["promo_intensity"]

    feats["streetwear_x_covid"] = feats["streetwear_share"] * int(date.year >= 2019)
    feats["casual_x_post2019"] = feats["casual_share"] * int(date.year >= 2019)
    feats["return_x_wrongsize"] = feats["return_flag"] * feats["wrong_size_rate"]
    feats["rating_x_return"] = feats["rating_mean"] * feats["return_flag"]
    feats["discount_x_mix"] = feats["discount_rate"] * feats["product_mix_entropy"]

    # base-specific interactions
    feats["base_x_covid"] = feats.get("base_raw", 0.0) * int(date.year >= 2019)
    feats["base_x_regime"] = feats.get("base_raw", 0.0) * regime

    return feats

def make_row(date, history):
    row = {}
    row.update(date_features(date))
    row.update(history_features(history))
    row.update(external_features(date))
    return row

def build_seq_frame(df_segment):
    df_segment = df_segment.sort_values("Date").reset_index(drop=True)
    history = []
    rows = []

    for _, r in df_segment.iterrows():
        date = r["Date"]
        feats = make_row(date, history)

        y = float(r["Revenue"])
        y_log = np.log1p(y)
        resid_log = y_log - feats["base_log"]

        feats["Date"] = date
        feats["Revenue"] = y
        feats["COGS"] = float(r["COGS"])
        feats["target_log"] = y_log
        feats["resid_log"] = resid_log

        rows.append(feats)
        history.append(y)

    return pd.DataFrame(rows)

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def fit_blend_weights(y_true, p1, p2, p3, step=0.05):
    best = None
    grid = np.arange(0, 1 + 1e-9, step)
    for w1 in grid:
        for w2 in grid:
            if w1 + w2 > 1:
                continue
            w3 = 1 - w1 - w2
            pred = w1 * p1 + w2 * p2 + w3 * p3
            mae = mean_absolute_error(y_true, pred)
            rmse = mean_squared_error(y_true, pred) ** 0.5
            r2 = r2_score(y_true, pred)
            cand = {"w1": w1, "w2": w2, "w3": w3, "mae": mae, "rmse": rmse, "r2": r2}
            if best is None or cand["mae"] < best["mae"]:
                best = cand
    return best

def recursive_forecast(dates, start_history, models, feature_cols, fill_median, weights=(1/3, 1/3, 1/3), cap_mult=1.08):
    history = list(start_history)
    dates = list(pd.to_datetime(dates))

    pred_lgb, pred_xgb, pred_ridge, pred_final = [], [], [], []
    base_series = []

    for date in dates:
        row = make_row(date, history)
        X_row = pd.DataFrame([row])[feature_cols].copy().fillna(fill_median)

        r1 = float(models["lgb"].predict(X_row)[0])
        r2 = float(models["xgb"].predict(X_row)[0])
        r3 = float(models["ridge"].predict(X_row)[0])

        resid = weights[0] * r1 + weights[1] * r2 + weights[2] * r3
        final_log = row["base_log"] + resid
        pred = float(np.expm1(final_log))
        pred = max(pred, 0.0)

        if len(history) >= 30:
            cap = max(np.max(history[-365:]), 1.0) * cap_mult
            pred = min(pred, cap)

        pred_lgb.append(max(float(np.expm1(row["base_log"] + r1)), 0.0))
        pred_xgb.append(max(float(np.expm1(row["base_log"] + r2)), 0.0))
        pred_ridge.append(max(float(np.expm1(row["base_log"] + r3)), 0.0))
        pred_final.append(pred)
        base_series.append(row["base_raw"])

        history.append(pred)

    return np.array(pred_lgb), np.array(pred_xgb), np.array(pred_ridge), np.array(pred_final), np.array(base_series)


# 6. BUILD DATA

print("\n" + "=" * 70)
print("STEP 5: BUILD FEATURES")
print("=" * 70)

train_feat = build_seq_frame(df_train)
val_feat   = build_seq_frame(df_val)
full_feat  = build_seq_frame(df_full)

FEATURE_COLS = [c for c in full_feat.columns if c not in ["Date", "Revenue", "COGS", "target_log", "resid_log"]]
X_train = train_feat[FEATURE_COLS].copy()
X_val   = val_feat[FEATURE_COLS].copy()
X_full  = full_feat[FEATURE_COLS].copy()

y_train_resid = train_feat["resid_log"].astype(float).copy()
y_val = val_feat["Revenue"].astype(float).copy()

fill_median = X_train.median(numeric_only=True)

X_train = X_train.fillna(fill_median)
X_val   = X_val.fillna(fill_median)
X_full  = X_full.fillna(fill_median)

# Smooth time weights: newer data matters more
train_years = train_feat["year"].values
full_years  = full_feat["year"].values
w_train = np.where(train_years < 2019, 0.20, np.where(train_years <= 2020, 0.70, 1.00))
w_full  = np.where(full_years  < 2019, 0.20, np.where(full_years  <= 2020, 0.70, 1.00))

print(f"  Features: {len(FEATURE_COLS)}")
print(f"  Missing rate: {X_train.isna().mean().mean():.4f}")


# 7. TRAIN RESIDUAL MODELS

print("\n" + "=" * 70)
print("STEP 6: TRAIN RESIDUAL MODELS")
print("=" * 70)

lgb_resid = lgb.LGBMRegressor(
    objective="huber",
    alpha=0.90,
    learning_rate=0.02,
    num_leaves=96,
    max_depth=7,
    min_child_samples=10,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=5,
    reg_alpha=0.08,
    reg_lambda=0.20,
    n_estimators=7000,
    random_state=SEED,
    verbose=-1,
)

xgb_resid = xgb.XGBRegressor(
    objective="reg:squarederror",
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=4,
    subsample=0.88,
    colsample_bytree=0.82,
    reg_alpha=0.05,
    reg_lambda=1.2,
    n_estimators=5000,
    tree_method="hist",
    random_state=SEED,
    verbosity=0,
)

ridge_resid = Pipeline([
    ("sc", StandardScaler()),
    ("m", Ridge(alpha=8.0))
])

tail_n = min(365, len(X_train))

lgb_resid.fit(
    X_train, y_train_resid,
    sample_weight=w_train,
    eval_set=[(X_train.tail(tail_n), y_train_resid.iloc[-tail_n:])],
    callbacks=[lgb.early_stopping(250, verbose=False), lgb.log_evaluation(-1)]
)

xgb_resid.fit(
    X_train, y_train_resid,
    sample_weight=w_train,
    eval_set=[(X_train.tail(tail_n), y_train_resid.iloc[-tail_n:])],
    verbose=False
)

ridge_resid.fit(X_train, y_train_resid, m__sample_weight=w_train)

print(f"  LGB best iteration: {getattr(lgb_resid, 'best_iteration_', None)}")


# 8. VALIDATION ON 2022

print("\n" + "=" * 70)
print("STEP 7: RECURSIVE VALIDATION ON 2022")
print("=" * 70)

start_history_val = df_train["Revenue"].astype(float).tolist()
val_dates = df_val["Date"].tolist()

p1_val, p2_val, p3_val, pblend_val, pbase_val = recursive_forecast(
    dates=val_dates,
    start_history=start_history_val,
    models={"lgb": lgb_resid, "xgb": xgb_resid, "ridge": ridge_resid},
    feature_cols=FEATURE_COLS,
    fill_median=fill_median,
    weights=(1/3, 1/3, 1/3),
    cap_mult=1.08
)

mae_lgb, rmse_lgb, r2_lgb = metrics(y_val.values, p1_val)
mae_xgb, rmse_xgb, r2_xgb = metrics(y_val.values, p2_val)
mae_ridge, rmse_ridge, r2_ridge = metrics(y_val.values, p3_val)

print(f"  LGB   -> MAE={mae_lgb:,.1f} | RMSE={rmse_lgb:,.1f} | R²={r2_lgb:.4f}")
print(f"  XGB   -> MAE={mae_xgb:,.1f} | RMSE={rmse_xgb:,.1f} | R²={r2_xgb:.4f}")
print(f"  Ridge -> MAE={mae_ridge:,.1f} | RMSE={rmse_ridge:,.1f} | R²={r2_ridge:.4f}")

best_w = fit_blend_weights(y_val.values, p1_val, p2_val, p3_val, step=0.05)
weights = (best_w["w1"], best_w["w2"], best_w["w3"])
pblend_val = weights[0] * p1_val + weights[1] * p2_val + weights[2] * p3_val

mae_blend, rmse_blend, r2_blend = metrics(y_val.values, pblend_val)
print(f"\n  Best blend: w_LGB={weights[0]:.2f}, w_XGB={weights[1]:.2f}, w_Ridge={weights[2]:.2f}")
print(f"  Blend -> MAE={mae_blend:,.1f} | RMSE={rmse_blend:,.1f} | R²={r2_blend:.4f}")

# Calibration uses the SAME recursive scheme as test
cal = Ridge(alpha=1.0, positive=True, random_state=SEED)
cal.fit(np.column_stack([pblend_val, pbase_val]), y_val.values)

cal_pred_val = cal.predict(np.column_stack([pblend_val, pbase_val]))
cal_pred_val = np.maximum(cal_pred_val, 0.0)
mae_cal, rmse_cal, r2_cal = metrics(y_val.values, cal_pred_val)

print(f"  Calibration -> MAE={mae_cal:,.1f} | RMSE={rmse_cal:,.1f} | R²={r2_cal:.4f}")
print(f"  Cal coef: {cal.coef_}, intercept={cal.intercept_:.4f}")


# 9. RETRAIN ON FULL 2012-2022

print("\n" + "=" * 70)
print("STEP 8: RETRAIN ON FULL 2012-2022")
print("=" * 70)

y_full_resid = full_feat["resid_log"].astype(float).copy()

lgb_final = lgb.LGBMRegressor(
    objective="huber",
    alpha=0.90,
    learning_rate=0.02,
    num_leaves=96,
    max_depth=7,
    min_child_samples=10,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=5,
    reg_alpha=0.08,
    reg_lambda=0.20,
    n_estimators=max(300, int(getattr(lgb_resid, "best_iteration_", 2500) or 2500)),
    random_state=SEED,
    verbose=-1,
)

xgb_params = xgb_resid.get_params().copy()
xgb_params.pop("random_state", None)
xgb_params.pop("verbosity", None)

xgb_final = xgb.XGBRegressor(
    **xgb_params,
    random_state=SEED,
    verbosity=0
)

ridge_final = Pipeline([
    ("sc", StandardScaler()),
    ("m", Ridge(alpha=8.0))
])

lgb_final.fit(
    X_full, y_full_resid,
    sample_weight=w_full,
    eval_set=[(X_full.tail(min(365, len(X_full))), y_full_resid.iloc[-min(365, len(X_full)):])],
    callbacks=[lgb.early_stopping(250, verbose=False), lgb.log_evaluation(-1)]
)
xgb_final.fit(X_full, y_full_resid, sample_weight=w_full)
ridge_final.fit(X_full, y_full_resid, m__sample_weight=w_full)


# 10. TEST FORECAST

print("\n" + "=" * 70)
print("STEP 9: RECURSIVE TEST FORECAST")
print("=" * 70)

start_history_test = df_full["Revenue"].astype(float).tolist()
test_dates = df_submit["Date"].tolist()

p1_test, p2_test, p3_test, pblend_test, pbase_test = recursive_forecast(
    dates=test_dates,
    start_history=start_history_test,
    models={"lgb": lgb_final, "xgb": xgb_final, "ridge": ridge_final},
    feature_cols=FEATURE_COLS,
    fill_median=fill_median,
    weights=weights,
    cap_mult=1.08
)

# Apply calibration learned on validation
p_final = cal.predict(np.column_stack([pblend_test, pbase_test]))
p_final = np.maximum(p_final, 0.0)

# Soft clip to avoid extreme runaway
recent_cap = max(np.max(df_full["Revenue"].tail(365).values), 1.0) * 1.08
p_final = np.clip(p_final, 0, recent_cap)

print(f"  Test predictions: mean={p_final.mean():,.1f} | std={p_final.std():,.1f} | max={p_final.max():,.1f}")

# COGS: keep historical margin structure
cogs_ratio = (df_full["COGS"] / df_full["Revenue"]).replace([np.inf, -np.inf], np.nan).dropna().median()
p_cogs = np.maximum(p_final * cogs_ratio, 0.0)
print(f"  COGS/Revenue median ratio: {cogs_ratio:.4f}")


# 11. SUBMISSION

print("\n" + "=" * 70)
print("STEP 10: SAVE SUBMISSION")
print("=" * 70)

sub = df_submit[["Date"]].copy()
sub["Revenue"] = np.round(p_final, 2)
sub["COGS"] = np.round(p_cogs, 2)
sub["Date"] = sub["Date"].dt.strftime("%Y-%m-%d")

out_path = f"{BASE}/submission.csv"
sub.to_csv(out_path, index=False)

print(f"   Saved: {out_path}")
print(sub.head(10).to_string(index=False))


# 12. SHAP

print("\n" + "=" * 70)
print("STEP 11: SHAP ANALYSIS")
print("=" * 70)

try:
    import shap
    sample_n = min(800, len(X_full))
    idx = np.random.choice(len(X_full), sample_n, replace=False)

    explainer = shap.TreeExplainer(lgb_final)
    shap_values = explainer.shap_values(X_full.iloc[idx])

    feat_imp = pd.DataFrame({
        "feature": FEATURE_COLS,
        "shap_mean": np.abs(shap_values).mean(axis=0),
        "lgb_imp": lgb_final.feature_importances_
    }).sort_values("shap_mean", ascending=False)

    print("\nTop 20 features by SHAP:")
    print(feat_imp.head(20)[["feature", "shap_mean", "lgb_imp"]].to_string(index=False))
except Exception as e:
    print(f"  SHAP skipped due to: {e}")
    feat_imp = pd.DataFrame({"feature": FEATURE_COLS, "shap_mean": np.nan, "lgb_imp": lgb_final.feature_importances_})


# 13. MODEL VISUALIZATION

print("\n" + "=" * 70)
print("STEP 12: MODEL VISUALIZATION")
print("=" * 70)

fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(2, 2, hspace=0.30, wspace=0.20)

# ── (1) Validation: actual vs predicted
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(df_val["Date"], y_val.values, linewidth=1.6, label="Actual")
ax1.plot(df_val["Date"], cal_pred_val, linewidth=1.6, linestyle="--", label="Predicted")
ax1.set_title(f"2022 Validation: Actual vs Predicted\nMAE={mae_cal:,.1f} | R²={r2_cal:.4f}")
ax1.set_xlabel("Date")
ax1.set_ylabel("Revenue")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis="x", rotation=25)

# ── (2) Scatter: predicted vs actual
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_val.values, cal_pred_val, alpha=0.25, s=10)
lims = [
    min(float(y_val.min()), float(cal_pred_val.min())) * 0.95,
    max(float(y_val.max()), float(cal_pred_val.max())) * 1.05
]
ax2.plot(lims, lims, linestyle="--")
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_title("Predicted vs Actual (Validation)")
ax2.set_xlabel("Actual Revenue")
ax2.set_ylabel("Predicted Revenue")
ax2.grid(True, alpha=0.3)

# ── (3) Residual distribution
ax3 = fig.add_subplot(gs[1, 0])
residuals = y_val.values - cal_pred_val
ax3.hist(residuals, bins=50, density=True, alpha=0.75)
ax3.axvline(0, linestyle="--")
ax3.axvline(residuals.mean(), linestyle=":")
ax3.set_title(
    f"Residual Distribution (Validation)\nMean={residuals.mean():,.1f} | Std={residuals.std():,.1f}"
)
ax3.set_xlabel("Residual (Actual - Predicted)")
ax3.set_ylabel("Density")
ax3.grid(True, alpha=0.3)

# ── (4) Forecast on test
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(df_full.tail(180)["Date"], df_full.tail(180)["Revenue"], linewidth=1.6, label="Last 180 days of train")
ax4.plot(df_submit["Date"], p_final, linewidth=1.6, linestyle="--", label="Forecast")
ax4.axvline(df_submit["Date"].min(), linestyle=":", label="Forecast start")
ax4.set_title("Test Forecast: Revenue")
ax4.set_xlabel("Date")
ax4.set_ylabel("Revenue")
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.tick_params(axis="x", rotation=25)

plt.tight_layout()
fig_path = f"{BASE}/report_model_forecast.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"   Saved: {fig_path}")


# 14. FINAL SUMMARY

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"  LGB   (2022): MAE={mae_lgb:>8,.1f} | RMSE={rmse_lgb:>8,.1f} | R²={r2_lgb:.4f}")
print(f"  XGB   (2022): MAE={mae_xgb:>8,.1f} | RMSE={rmse_xgb:>8,.1f} | R²={r2_xgb:.4f}")
print(f"  Ridge (2022): MAE={mae_ridge:>8,.1f} | RMSE={rmse_ridge:>8,.1f} | R²={r2_ridge:.4f}")
print(f"  Blend (2022): MAE={mae_blend:>8,.1f} | RMSE={rmse_blend:>8,.1f} | R²={r2_blend:.4f}")
print(f"  Calib (2022): MAE={mae_cal:>8,.1f} | RMSE={rmse_cal:>8,.1f} | R²={r2_cal:.4f}")
print(f"  Final submission saved to: {out_path}")

STEP 1: LOAD DATA
  Train (<=2021): 3,468
  Val   (2022)  : 365
  Full  (<=2022): 3,833
  Test skeleton : 548

STEP 2: EXTERNAL LOOKUPS
   External signals ready

STEP 3: BUSINESS PROFILES
   Business profiles ready

STEP 4: FEATURE HELPERS

STEP 5: BUILD FEATURES
  Features: 200
  Missing rate: 0.0000

STEP 6: TRAIN RESIDUAL MODELS
  LGB best iteration: 4945

STEP 7: RECURSIVE VALIDATION ON 2022
  LGB   -> MAE=674,649.1 | RMSE=912,416.4 | R²=0.7029
  XGB   -> MAE=675,884.5 | RMSE=913,033.4 | R²=0.7024
  Ridge -> MAE=682,726.9 | RMSE=928,665.6 | R²=0.6922

  Best blend: w_LGB=0.45, w_XGB=0.45, w_Ridge=0.10
  Blend -> MAE=667,640.8 | RMSE=905,193.7 | R²=0.7075
  Calibration -> MAE=594,483.4 | RMSE=811,951.8 | R²=0.7647
  Cal coef: [1.01216441 0.        ], intercept=365628.9006

STEP 8: RETRAIN ON FULL 2012-2022

STEP 9: RECURSIVE TEST FORECAST
  Test predictions: mean=3,663,986.0 | std=1,571,821.7 | max=10,968,678.4
  COGS/Revenue median ratio: 0.8217

STEP 10: SAVE SUBMISSION
   Saved:

3. Phương pháp tiếp cận, pipeline mô hình, kết quả thực nghiệm và các yếu tố dẫn động doanh thu
3.1. Phương pháp tiếp cận

Bài toán được xây dựng dưới dạng dự báo chuỗi thời gian có yếu tố ngoại sinh. Mô hình không dự báo trực tiếp doanh thu mà kết hợp giữa baseline theo mùa vụ và phương pháp residual learning.

Cụ thể, baseline được tính dựa trên các quy luật như cùng kỳ năm trước và các chu kỳ tuần, tháng, quý, nhằm phản ánh xu hướng tự nhiên của doanh thu. Sau đó, các mô hình machine learning học phần sai lệch giữa thực tế và baseline, tập trung vào các yếu tố phức tạp như khuyến mãi, traffic và tồn kho. Ngoài ra, phương pháp recursive forecasting được áp dụng để mô phỏng quá trình dự báo nhiều bước trong thực tế.

3.2. Pipeline mô hình

Pipeline gồm ba bước chính.

Đầu tiên, dữ liệu được tích hợp từ nhiều nguồn (sales, traffic, promotions, inventory, …) và đồng bộ theo thời gian ở mức ngày.

Tiếp theo, các đặc trưng được xây dựng bao gồm: đặc trưng thời gian (tháng, ngày, mùa vụ), đặc trưng lịch sử (lag, rolling, growth), baseline mùa vụ, và các biến ngoại sinh như traffic, khuyến mãi, tồn kho và hành vi khách hàng.

Cuối cùng, ba mô hình LightGBM, XGBoost và Ridge được sử dụng để học residual. Kết quả được kết hợp bằng ensemble và hiệu chỉnh (calibration) để giảm bias và tăng độ ổn định. Dự báo được triển khai theo cơ chế recursive cho bài toán nhiều bước.

3.3. Kết quả thực nghiệm

Mô hình được đánh giá trên tập validation năm 2022 bằng MAE, RMSE và R². Kết quả cho thấy LightGBM đạt hiệu suất tốt nhất trong các mô hình đơn, trong khi ensemble giúp cải thiện độ chính xác tổng thể.

Bước calibration tiếp tục giảm sai lệch và ổn định dự báo. Mô hình cuối cùng đạt sai số thấp và R² cao, đồng thời bám sát xu hướng thực tế, kể cả trong các giai đoạn biến động mạnh. Phân phối phần dư tập trung quanh 0, cho thấy mô hình không bị bias đáng kể.

3.4. Các yếu tố dẫn động doanh thu

Phân tích feature importance và SHAP cho thấy ba nhóm yếu tố chính ảnh hưởng đến doanh thu.

Thứ nhất, xu hướng quá khứ (lag, rolling, year-over-year) có tác động lớn nhất, cho thấy doanh thu có tính quán tính cao.

Thứ hai, mùa vụ và thời điểm (tháng, ngày trong tuần, ngày lễ) giúp mô hình nắm bắt các chu kỳ lặp lại trong năm.

Thứ ba, yếu tố vận hành – thương mại như traffic, khuyến mãi và tồn kho đóng vai trò quan trọng. Traffic phản ánh nhu cầu khách hàng, khuyến mãi kích cầu, còn tồn kho ảnh hưởng đến khả năng đáp ứng.

Tổng thể, doanh thu được dẫn dắt bởi sự kết hợp giữa xu hướng lịch sử, mùa vụ và các yếu tố kinh doanh thực tế, giúp mô hình vừa dự báo chính xác vừa có ý nghĩa giải thích.